# Heath–Jarrow–Morton (HJM) Term Structure Modeling of U.S. Interest Rates
Author: Maximilian Yap, Cornell University

<h1>Table of Contents<span class="tocSkip"></span></h1>



## 0. Front Matter & Reproducibility

**Last Updated:**  
12-Jan-2025

**Repository:**  
<[GitHub repository link](https://github.com/mmy32/HJM-Libor-Model)>  
**Commit hash:** <short hash>

**Environment Specification:**  
- Python: 3.9+  
- Core libraries: `numpy`, `pandas`, `scipy`, `scikit-learn`, `plotly`, `statsmodels`  
- Optional: `dvc` for data versioning

**Execution Contract:**  
This notebook is designed to be executed **top-to-bottom** without manual intervention.  
All parameters controlling calibration windows, tenors, factor counts, and simulation horizons are defined explicitly in Section 4.

### What This Notebook Produces
- A clean, reproducible pipeline from raw yield data to:
  - factor-extracted curve dynamics,
  - arbitrage-consistent HJM drift and volatility terms,
  - simulated future yield curve scenarios.

**Expected Runtime:**  
~30 minutes on a standard laptop for calibration and diagnostics.


## 1. Executive Summary

### Objective
The goal of this project is to **model, calibrate, and simulate the evolution of the interest rate term structure** using a Heath–Jarrow–Morton (HJM) framework calibrated to historical U.S. Treasury yield data.



### Modeling Assumptions (High-Level)
- Interest rate dynamics are adequately captured by a **low-dimensional factor structure**.
- Historical yield movements are informative about future volatility (stationarity assumption).
- No-arbitrage conditions are enforced through the HJM drift restriction.


## 2. Introduction: Yield Curve Modeling and the HJM Framework

### 2.1 The Problem of Yield Curve Fitting
Interest rates are observed at a discrete set of maturities, yet many financial applications, such as risk management, scenario analysis, and pricing, require a **continuous, arbitrage-consistent representation of the entire yield curve** as it evolves over time. Empirically, yield curves must satisfy several competing requirements: they should fit observed market data closely, evolve smoothly across maturities, and generate realistic dynamics over time. Naïve fitting approaches often fail one or more of these criteria, leading to unstable extrapolations, implausible curve shapes, or violations of no-arbitrage conditions.


The Heath–Jarrow–Morton (HJM) framework provides a theoretically rigorous solution by modeling the **entire forward rate curve as a stochastic process**. Rather than specifying dynamics for a single short rate, HJM directly characterizes the evolution of forward rates across maturities. Crucially, once the volatility structure of the forward curve is specified, the drift is uniquely determined by a no-arbitrage condition. This makes HJM a natural framework for generating arbitrage-free yield curve dynamics and coherent multi-maturity scenarios.

### 2.2 Limitations of a Naïve HJM Implementation
Despite its theoretical appeal, a basic HJM implementation faces several practical challenges:
- The forward curve is infinite-dimensional, making unrestricted volatility specification infeasible.
- Arbitrary volatility choices can lead to unstable or economically implausible dynamics.
- Estimating a high-dimensional volatility structure directly from data is noisy and prone to overfitting.
- Without dimensionality reduction, simulations become computationally expensive and difficult to interpret.

These issues limit the usefulness of HJM unless additional structure is imposed.

### 2.3 PCA as a Practical Resolution

Empirically, yield curve movements are highly correlated and can be well-approximated by a **small number of common factors**. Principal Component Analysis (PCA) exploits this structure by identifying the dominant modes of variation—commonly interpreted as level, slope, and curvature effects. By projecting historical yield (or forward rate) changes onto a low-dimensional factor space, PCA provides:
- a parsimonious and data-driven volatility specification,
- noise reduction and improved stability,
- interpretable economic dynamics.

Embedding these PCA-derived factors into the HJM framework yields a **low-dimensional, arbitrage-consistent model** that remains faithful to observed yield curve behavior.

### 2.5 Process Overview and Intended Usefulness
This notebook implements the following pipeline:
1. ingest and clean historical yield curve data,
2. extract dominant factors via PCA,
3. map factor volatilities into an HJM-consistent volatility structure,
4. compute the implied no-arbitrage drift,
5. simulate future yield curve scenarios and validate their properties.

The ultimate objective is to produce a **transparent, reproducible, and interpretable yield curve model** that balances theoretical soundness with empirical realism. While not intended as a production pricing system, the framework is designed to be a useful foundation for scenario generation, stress testing, and further extensions in quantitative fixed-income research.



## 3. Data & Provenance

The data source of this project is a panel of U.S. Treasury constant maturity yields obtained directly from the Federal Reserve Economic Data (FRED) database.

Raw fetching, cleaning, and persistence are three separate, composable steps rather than one function that does everything silently:
- `src/data_processing/loaders.py::fetch_treasury_yields` queries a set of standard Treasury yield series identified by their FRED symbols (e.g., DGS1MO, DGS2, DGS10) -- defined in `src/registry/market_data.py` -- and returns the raw panel, exactly as reported by FRED (percentage terms, no gap handling).
- `src/data_processing/cleaning.py::clean_treasury_yields` converts percentage yields to decimals, forward-fills short gaps, and drops any row still incomplete afterward. Forward-filling assumes an unobserved quote is unchanged from the prior observation; this avoids introducing artificial noise through interpolation while preserving the joint cross-sectional structure of the curve.
- `src/data_processing/io.py::save_yield_matrix` / `load_yield_matrix` handle the CSV read/write to `data/treasury_yields.csv`, the canonical input for the remainder of the project.

The cell below composes these three steps explicitly, generating the yield matrix within the notebook environment and returning it as a pandas DataFrame. Successful execution confirms that the data required for the HJM calibration pipeline have been built correctly and are available for further analysis.


In [1]:
import sys
from pathlib import Path

# If your notebook is in ./notebooks, this points to repo root
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").exists() is False:
    PROJECT_ROOT = Path.cwd().parent  # adjust if needed

sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.data_processing.loaders import fetch_treasury_yields
from src.data_processing.cleaning import clean_treasury_yields
from src.data_processing.io import save_yield_matrix

raw = fetch_treasury_yields(start_date="2018-01-01")
df = clean_treasury_yields(raw)
save_yield_matrix(df)
df.head()

After execution, the DataFrame index consists of observation dates, the columns correspond to maturities expressed as floating-point numbers in years, and the entries contain Treasury yields in decimal form. This representation provides a direct and transparent link between the raw market data and the stochastic term-structure model developed in the subsequent sections.


In [ ]:
from src.viz.curves import build_yield_curve_slider_figure

build_yield_curve_slider_figure(df).show()

## 4. Nelson-Siegel Curve Fitting

The raw yield panel is observed at only 11 discrete tenors. To get a continuous curve on every date, each day's cross-section is fit to the Nelson-Siegel parametric form (`src/curves/nelson_siegel.py`), which reduces each day's curve to four interpretable parameters: level (`b0`), slope (`b1`), curvature (`b2`), and decay (`lambda`).

`nelson_siegel_yield` and `nelson_siegel_forward` are the two canonical, mutually-consistent implementations of the model (the forward curve is the instantaneous rate; the yield curve is its running average) -- earlier drafts of this project had three independent, silently inconsistent copies of the forward-rate formula scattered across files, which is exactly the kind of bug this reorganization is meant to prevent. `fit_ns_robust` calibrates one day via global optimization against bounds defined once in `src/registry/curve_spec.py`; `calibrate_all_days` applies it across the whole panel.

In [ ]:
import numpy as np

from src.curves.nelson_siegel import calibrate_all_days
from src.persistence import artifacts
from src.registry.curve_spec import SMOOTH_GRID_MAX, SMOOTH_GRID_N

tenors = np.array([float(c) for c in df.columns])
smooth_tenors = np.linspace(0, SMOOTH_GRID_MAX, SMOOTH_GRID_N)

ns_params_df = calibrate_all_days(df, tenors, progress=True)
artifacts.save_ns_parameters(ns_params_df)
ns_params_df.describe()

In [ ]:
from src.viz.curves import build_ns_fit_slider_figure

build_ns_fit_slider_figure(df, ns_params_df, tenors, smooth_tenors, sample_every=15).show()

from src.calibration.pca import fit_pca
from src.viz.pca import build_pca_diagnostics_figure

pca_model = fit_pca(ns_params_df)
artifacts.save_pca_result(pca_model)

print(pca_model.explained_variance_ratio)
build_pca_diagnostics_figure(pca_model)

## 6. Estimating Principal Component (PC) Processes

Each PC score series is modeled as a mean-reverting Ornstein-Uhlenbeck process: `dX = kappa(theta - X)dt + sigma dW`. `src/calibration/ou_process.py::estimate_ou_parameters` fits `(kappa, theta, sigma)` per factor via maximum likelihood; `estimate_ou_parameters_for_factors` applies it across every PC column at once. These per-factor `(kappa, theta, sigma)` triples become the volatility and mean-reversion inputs to the HJM simulator in Section 8.

In [ ]:
import pandas as pd

from src.calibration.ou_process import estimate_ou_parameters_for_factors
from src.viz.ou import build_ou_diagnostics_figure
from src.viz.style import apply_default_style

apply_default_style()

ou_params = estimate_ou_parameters_for_factors(pca_model.scores)
artifacts.save_ou_parameters(ou_params)

display(pd.DataFrame(ou_params).T)
build_ou_diagnostics_figure(pca_model.scores, ou_params)

## 7. Computing Nelson-Siegel Sensitivities

To connect PC movements to forward-rate movements, we need the chain rule: how does the forward rate at each maturity respond to a one-unit move in each NS parameter (`ns_sensitivities`), and then how does that translate into a response to each PC (`compute_forward_sensitivities`, which combines the NS sensitivities with the PCA loadings from Section 5)? Both live in `src/calibration/sensitivities.py`, built directly on the canonical `nelson_siegel_forward` from Section 4 rather than a separate reimplementation.

In [ ]:
from src.calibration.sensitivities import compute_forward_sensitivities, ns_sensitivities
from src.registry.curve_spec import MATURITY_GRID
from src.viz.sensitivities import build_ns_sensitivities_figure, build_pc_sensitivities_figure

mean_params = ns_params_df.mean().to_dict()

sens = ns_sensitivities(
    MATURITY_GRID, mean_params["b0_level"], mean_params["b1_slope"], mean_params["b2_curvature"], mean_params["lambda"]
)
build_ns_sensitivities_figure(MATURITY_GRID, sens)

In [ ]:
pc_sens_df = compute_forward_sensitivities(mean_params, pca_model.loadings, MATURITY_GRID)
artifacts.save_sensitivities(mean_params, MATURITY_GRID, pc_sens_df)

build_pc_sensitivities_figure(pc_sens_df)

from src.stochastic.hjm_model import HJMModel

hjm_model = HJMModel.from_disk()

results_P = hjm_model.simulate(n_paths=1000, T_horizon=0.25, dt=1 / 252, measure="P", random_seed=42)
results_Q = hjm_model.simulate(n_paths=1000, T_horizon=1.0, dt=1 / 52, measure="Q", random_seed=42)

idx_10y = int(np.argmin(np.abs(results_P.maturities - 10.0)))
print(f"P-measure 10Y rate at T=0.25y: mean={results_P.zero_curves[:, -1, idx_10y].mean() * 100:.2f}%")
print(f"Q-measure 10Y rate at T=1.0y:  mean={results_Q.zero_curves[:, -1, idx_10y].mean() * 100:.2f}%")

In [ ]:
from src.stochastic.hjm_model import HJMModel

hjm_model = HJMModel.from_disk()

results_P = hjm_model.simulate(n_paths=1000, T_horizon=0.25, dt=1 / 252, measure="P", random_seed=42)
results_Q = hjm_model.simulate(n_paths=1000, T_horizon=1.0, dt=1 / 52, measure="Q", random_seed=42)

print(f"P-measure 10Y rate at T=0.25y: mean={results_P.zero_curves[:, -1, -1].mean() * 100:.2f}%")
print(f"Q-measure 10Y rate at T=1.0y:  mean={results_Q.zero_curves[:, -1, -1].mean() * 100:.2f}%")

In [ ]:
from src.viz.simulation import build_sample_paths_figure

build_sample_paths_figure(results_P, n_factors=hjm_model.n_factors, n_sample=10, random_seed=0)

In [ ]:
build_sample_paths_figure(results_Q, n_factors=hjm_model.n_factors, n_sample=10, random_seed=0)

## 9. Summary

This notebook now runs the full pipeline end-to-end -- data ingestion, Nelson-Siegel curve fitting, PCA factor extraction, OU process estimation, sensitivity computation, and HJM simulation under both measures -- with every stage's actual logic living in `src/` as pure, independently unit-tested functions (see `tests/`), and this notebook doing nothing but composing and visualizing them. See `TODO.md` for open items, including calibrating a real market price of risk for the Q-measure and fully vectorizing `HJMModel.simulate` across paths.

In [ ]:
from src.calibration.pca import fit_pca
from src.viz.pca import build_pca_diagnostics_figure

pca_model = fit_pca(ns_params_df)
artifacts.save_pca_result(pca_model)

build_pca_diagnostics_figure(pca_model)
pca_model.explained_variance_ratio